# 🚀 GPU-Accelerated High-Volume Sepsis Extraction

## Target: 50,000+ Samples

**Run cells in order (0 → 10)**

In [1]:
# ============================================================================
# CELL 0: SETUP - Install & Authenticate
# ============================================================================

!pip install -q google-cloud-bigquery db-dtypes pyarrow tqdm joblib

from google.colab import auth
auth.authenticate_user()

print("✅ Setup complete!")

✅ Setup complete!


In [2]:
# ============================================================================
# CELL 1: CONFIGURATION
# ============================================================================

import numpy as np
import pandas as pd
from google.cloud import bigquery
import torch
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import cpu_count
from joblib import Parallel, delayed
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# YOUR PROJECT ID
# ============================================================================
PROJECT_ID = 'sepsis-prediction-2025'

# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    'target_samples': 50000,
    'control_ratio': 5,
    'min_icu_hours': 12,
    'min_data_hours': 4,
    'prediction_gap_hours': 2,
    'sofa_increase_threshold': 2,
    'observation_window_hours': 24,
    'chunk_size': 5000,
    'use_gpu': torch.cuda.is_available(),
    'project_id': PROJECT_ID,
}

DEVICE = torch.device('cuda' if CONFIG['use_gpu'] else 'cpu')
N_CORES = cpu_count()

print(f"{'='*60}")
print("CONFIGURATION")
print(f"{'='*60}")
print(f"  Project ID: {CONFIG['project_id']}")
print(f"  Target: {CONFIG['target_samples']:,} samples")
print(f"  Control ratio: 1:{CONFIG['control_ratio']}")
print(f"  GPU: {DEVICE}")
print(f"  CPU Cores: {N_CORES}")
print(f"{'='*60}")

# Initialize BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Test connection
test = client.query("SELECT 1 as test").to_dataframe()
print("\n✅ BigQuery connection successful!")

CONFIGURATION
  Project ID: sepsis-prediction-2025
  Target: 50,000 samples
  Control ratio: 1:5
  GPU: cuda
  CPU Cores: 12

✅ BigQuery connection successful!


In [3]:
# ============================================================================
# CELL 2: GET ICU STAYS
# ============================================================================

print("="*70)
print("HIGH-VOLUME DATA EXTRACTION")
print("="*70)

print(f"\n[1/8] Getting ICU stays (min {CONFIG['min_icu_hours']}h)...")

query_stays = f"""
SELECT
    icu.stay_id,
    icu.subject_id,
    icu.hadm_id,
    icu.intime,
    icu.outtime,
    TIMESTAMP_DIFF(icu.outtime, icu.intime, HOUR) as los_hours,
    pat.gender,
    pat.anchor_age as age,
    ROW_NUMBER() OVER (PARTITION BY icu.subject_id ORDER BY icu.intime) as stay_num
FROM `physionet-data.mimiciv_3_1_icu.icustays` icu
JOIN `physionet-data.mimiciv_3_1_hosp.patients` pat
    ON icu.subject_id = pat.subject_id
WHERE TIMESTAMP_DIFF(icu.outtime, icu.intime, HOUR) >= {CONFIG['min_icu_hours']}
ORDER BY icu.subject_id, icu.intime
"""

stays_df = client.query(query_stays).to_dataframe()
print(f"   ✅ Total ICU stays: {len(stays_df):,}")
print(f"   ✅ Unique patients: {stays_df['subject_id'].nunique():,}")

HIGH-VOLUME DATA EXTRACTION

[1/8] Getting ICU stays (min 12h)...
   ✅ Total ICU stays: 90,776
   ✅ Unique patients: 63,671


In [4]:
# ============================================================================
# CELL 3: GET SUSPECTED INFECTIONS
# ============================================================================

print(f"\n[2/8] Identifying suspected infections...")

query_infection = """
WITH antibiotics AS (
    SELECT DISTINCT
        ie.stay_id,
        ie.subject_id,
        MIN(ie.starttime) as abx_time
    FROM `physionet-data.mimiciv_3_1_icu.inputevents` ie
    WHERE ie.itemid IN (
        225798, 225842, 225843, 225844, 225845, 225846, 225847,
        225848, 225849, 225850, 225851, 225853, 225855, 225857,
        225859, 225860, 225862, 225863, 225865, 225866, 225868,
        225869, 225871, 225873, 225875, 225876, 225877, 225879,
        225881, 225882, 225883, 225884, 225885, 225886, 225888,
        225889, 225890, 225892, 225893, 225895, 225896, 225897,
        225898, 225899, 227689
    )
    GROUP BY ie.stay_id, ie.subject_id
),
cultures AS (
    SELECT DISTINCT
        icu.stay_id,
        icu.subject_id,
        MIN(me.charttime) as culture_time
    FROM `physionet-data.mimiciv_3_1_hosp.microbiologyevents` me
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu
        ON me.subject_id = icu.subject_id
        AND me.charttime BETWEEN icu.intime AND icu.outtime
    GROUP BY icu.stay_id, icu.subject_id
)
SELECT
    COALESCE(a.stay_id, c.stay_id) as stay_id,
    COALESCE(a.subject_id, c.subject_id) as subject_id,
    a.abx_time,
    c.culture_time,
    LEAST(
        COALESCE(a.abx_time, c.culture_time),
        COALESCE(c.culture_time, a.abx_time)
    ) as suspected_infection_time
FROM antibiotics a
FULL OUTER JOIN cultures c ON a.stay_id = c.stay_id
WHERE a.stay_id IS NOT NULL OR c.stay_id IS NOT NULL
"""

infection_df = client.query(query_infection).to_dataframe()
print(f"   ✅ Stays with suspected infection: {len(infection_df):,}")


[2/8] Identifying suspected infections...
   ✅ Stays with suspected infection: 74,591


In [5]:
# ============================================================================
# CELL 4: GET SOFA COMPONENT DATA
# ============================================================================

print(f"\n[3/8] Extracting SOFA components...")

infected_stays = infection_df['stay_id'].dropna().astype(int).tolist()
print(f"   Processing {len(infected_stays):,} infected stays...")

def get_sofa_chunk(stay_ids_chunk):
    stays_str = ','.join(map(str, stay_ids_chunk))
    query = f"""
    SELECT stay_id, charttime, 'pao2' as component, valuenum as value
    FROM `physionet-data.mimiciv_3_1_icu.chartevents`
    WHERE itemid IN (220224, 490) AND valuenum IS NOT NULL AND stay_id IN ({stays_str})
    UNION ALL
    SELECT stay_id, charttime, 'fio2' as component,
           CASE WHEN valuenum > 1 THEN valuenum/100.0 ELSE valuenum END as value
    FROM `physionet-data.mimiciv_3_1_icu.chartevents`
    WHERE itemid IN (223835, 190) AND valuenum IS NOT NULL AND stay_id IN ({stays_str})
    UNION ALL
    SELECT stay_id, charttime, 'map' as component, valuenum as value
    FROM `physionet-data.mimiciv_3_1_icu.chartevents`
    WHERE itemid = 220052 AND valuenum IS NOT NULL AND stay_id IN ({stays_str})
    UNION ALL
    SELECT stay_id, charttime, 'gcs' as component, valuenum as value
    FROM `physionet-data.mimiciv_3_1_icu.chartevents`
    WHERE itemid IN (223900, 223901, 220739) AND valuenum IS NOT NULL AND stay_id IN ({stays_str})
    UNION ALL
    SELECT icu.stay_id, le.charttime, 'platelets' as component, le.valuenum as value
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu
        ON le.subject_id = icu.subject_id AND le.charttime BETWEEN icu.intime AND icu.outtime
    WHERE le.itemid = 51265 AND le.valuenum IS NOT NULL AND icu.stay_id IN ({stays_str})
    UNION ALL
    SELECT icu.stay_id, le.charttime, 'bilirubin' as component, le.valuenum as value
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu
        ON le.subject_id = icu.subject_id AND le.charttime BETWEEN icu.intime AND icu.outtime
    WHERE le.itemid = 50885 AND le.valuenum IS NOT NULL AND icu.stay_id IN ({stays_str})
    UNION ALL
    SELECT icu.stay_id, le.charttime, 'creatinine' as component, le.valuenum as value
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu
        ON le.subject_id = icu.subject_id AND le.charttime BETWEEN icu.intime AND icu.outtime
    WHERE le.itemid = 50912 AND le.valuenum IS NOT NULL AND icu.stay_id IN ({stays_str})
    """
    return client.query(query).to_dataframe()

chunk_size = 8000
sofa_dfs = []

for i in tqdm(range(0, len(infected_stays), chunk_size), desc="   SOFA chunks"):
    chunk = infected_stays[i:i+chunk_size]
    try:
        df = get_sofa_chunk(chunk)
        if len(df) > 0:
            sofa_dfs.append(df)
    except Exception as e:
        print(f"   ⚠️ Chunk {i//chunk_size} warning: {str(e)[:50]}")
        continue

sofa_data = pd.concat(sofa_dfs, ignore_index=True) if sofa_dfs else pd.DataFrame()
print(f"\n   ✅ SOFA measurements: {len(sofa_data):,}")
print(f"   ✅ Stays with SOFA data: {sofa_data['stay_id'].nunique():,}")


[3/8] Extracting SOFA components...
   Processing 74,591 infected stays...


   SOFA chunks: 100%|██████████| 10/10 [00:45<00:00,  4.58s/it]



   ✅ SOFA measurements: 11,375,237
   ✅ Stays with SOFA data: 74,554


In [6]:
# ============================================================================
# CELL 5: CALCULATE SOFA SCORES (GPU-ACCELERATED)
# ============================================================================

print(f"\n[4/8] Computing SOFA scores (GPU-accelerated)...")

# Move data to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"   Using device: {device}")

def compute_sofa_gpu_batch(stay_ids_batch, sofa_data, stays_info):
    """Compute SOFA scores for a batch of stays using GPU."""
    results = []

    for stay_id in stay_ids_batch:
        if stay_id not in stays_info:
            continue

        stay_data = sofa_data[sofa_data['stay_id'] == stay_id].copy()
        if len(stay_data) == 0:
            continue

        intime = stays_info[stay_id]
        stay_data['hours'] = (stay_data['charttime'] - intime).dt.total_seconds() / 3600
        stay_data['hour_bin'] = stay_data['hours'].astype(int)

        # Pivot data
        hourly = stay_data.groupby(['hour_bin', 'component'])['value'].mean().unstack(fill_value=np.nan)
        hourly = hourly.ffill().bfill()

        if len(hourly) == 0:
            continue

        # Convert to GPU tensors
        hours = torch.tensor(hourly.index.values, device=device, dtype=torch.float32)

        # Initialize component tensors (fill with NaN-safe defaults)
        n_hours = len(hours)

        pao2 = torch.zeros(n_hours, device=device)
        fio2 = torch.ones(n_hours, device=device)  # Default 1 to avoid div by zero
        platelets = torch.full((n_hours,), 200.0, device=device)
        bilirubin = torch.zeros(n_hours, device=device)
        map_val = torch.full((n_hours,), 80.0, device=device)
        gcs = torch.full((n_hours,), 15.0, device=device)
        creatinine = torch.zeros(n_hours, device=device)

        # Fill with actual values
        if 'pao2' in hourly.columns:
            pao2 = torch.tensor(hourly['pao2'].fillna(0).values, device=device, dtype=torch.float32)
        if 'fio2' in hourly.columns:
            fio2 = torch.tensor(hourly['fio2'].fillna(1).values, device=device, dtype=torch.float32)
            fio2 = torch.clamp(fio2, min=0.01)  # Avoid division by zero
        if 'platelets' in hourly.columns:
            platelets = torch.tensor(hourly['platelets'].fillna(200).values, device=device, dtype=torch.float32)
        if 'bilirubin' in hourly.columns:
            bilirubin = torch.tensor(hourly['bilirubin'].fillna(0).values, device=device, dtype=torch.float32)
        if 'map' in hourly.columns:
            map_val = torch.tensor(hourly['map'].fillna(80).values, device=device, dtype=torch.float32)
        if 'gcs' in hourly.columns:
            gcs = torch.tensor(hourly['gcs'].fillna(15).values, device=device, dtype=torch.float32)
        if 'creatinine' in hourly.columns:
            creatinine = torch.tensor(hourly['creatinine'].fillna(0).values, device=device, dtype=torch.float32)

        # GPU-vectorized SOFA calculation
        sofa = torch.zeros(n_hours, device=device)

        # Respiratory (PaO2/FiO2)
        pf_ratio = pao2 / fio2
        sofa += torch.where(pf_ratio < 100, 4.0,
                torch.where(pf_ratio < 200, 3.0,
                torch.where(pf_ratio < 300, 2.0,
                torch.where(pf_ratio < 400, 1.0, 0.0))))

        # Coagulation (Platelets)
        sofa += torch.where(platelets < 20, 4.0,
                torch.where(platelets < 50, 3.0,
                torch.where(platelets < 100, 2.0,
                torch.where(platelets < 150, 1.0, 0.0))))

        # Liver (Bilirubin)
        sofa += torch.where(bilirubin >= 12, 4.0,
                torch.where(bilirubin >= 6, 3.0,
                torch.where(bilirubin >= 2, 2.0,
                torch.where(bilirubin >= 1.2, 1.0, 0.0))))

        # Cardiovascular (MAP)
        sofa += torch.where(map_val < 70, 1.0, 0.0)

        # Neurological (GCS)
        sofa += torch.where(gcs < 6, 4.0,
                torch.where(gcs < 10, 3.0,
                torch.where(gcs < 13, 2.0,
                torch.where(gcs < 15, 1.0, 0.0))))

        # Renal (Creatinine)
        sofa += torch.where(creatinine >= 5.0, 4.0,
                torch.where(creatinine >= 3.5, 3.0,
                torch.where(creatinine >= 2.0, 2.0,
                torch.where(creatinine >= 1.2, 1.0, 0.0))))

        # Move back to CPU and create results
        sofa_cpu = sofa.cpu().numpy()
        hours_cpu = hours.cpu().numpy()

        for h, s in zip(hours_cpu, sofa_cpu):
            results.append({'stay_id': stay_id, 'hour': int(h), 'sofa': float(s)})

    return results

# Process in batches on GPU
stays_info = stays_df.set_index('stay_id')['intime'].to_dict()
unique_stays = sofa_data['stay_id'].unique()

print(f"   Processing {len(unique_stays):,} stays on GPU...")

batch_size = 500
all_results = []

for i in tqdm(range(0, len(unique_stays), batch_size), desc="   GPU SOFA"):
    batch = unique_stays[i:i+batch_size]
    batch_results = compute_sofa_gpu_batch(batch, sofa_data, stays_info)
    all_results.extend(batch_results)

all_sofa_df = pd.DataFrame(all_results)

print(f"\n   ✅ SOFA scores: {len(all_sofa_df):,}")
print(f"   ✅ Stays with SOFA: {all_sofa_df['stay_id'].nunique():,}")


[4/8] Computing SOFA scores (GPU-accelerated)...
   Using device: cuda
   Processing 74,554 stays on GPU...


   GPU SOFA: 100%|██████████| 150/150 [22:55<00:00,  9.17s/it]



   ✅ SOFA scores: 4,288,104
   ✅ Stays with SOFA: 73,068


In [7]:
# ============================================================================
# CELL 6: IDENTIFY SEPSIS CASES
# ============================================================================

print(f"\n[5/8] Identifying Sepsis-3 cases...")

def identify_sepsis(stay_id, sofa_df, min_data_hours):
    stay_sofa = sofa_df[sofa_df['stay_id'] == stay_id].sort_values('hour')

    if len(stay_sofa) < 2:
        return None

    baseline = stay_sofa[stay_sofa['hour'] <= 6]['sofa'].min()
    if pd.isna(baseline):
        baseline = stay_sofa['sofa'].iloc[0]

    for _, row in stay_sofa.iterrows():
        if row['sofa'] >= baseline + CONFIG['sofa_increase_threshold']:
            onset_hour = row['hour']
            if onset_hour >= min_data_hours + CONFIG['prediction_gap_hours']:
                return {
                    'stay_id': stay_id,
                    'onset_hour': onset_hour,
                    'baseline_sofa': baseline,
                    'onset_sofa': row['sofa']
                }
    return None

infected_set = set(infection_df['stay_id'].dropna().astype(int))
sofa_stays = all_sofa_df['stay_id'].unique()
eligible_stays = [s for s in sofa_stays if s in infected_set]

print(f"   Checking {len(eligible_stays):,} eligible stays...")

sepsis_results = Parallel(n_jobs=N_CORES, backend='threading')(
    delayed(identify_sepsis)(sid, all_sofa_df, CONFIG['min_data_hours'])
    for sid in tqdm(eligible_stays, desc="   Sepsis ID")
)

sepsis_cases = [r for r in sepsis_results if r is not None]
sepsis_df = pd.DataFrame(sepsis_cases)

print(f"\n   ✅ Sepsis-3 cases: {len(sepsis_df):,}")
if len(sepsis_df) > 0:
    print(f"   Mean onset hour: {sepsis_df['onset_hour'].mean():.1f}")


[5/8] Identifying Sepsis-3 cases...
   Checking 73,068 eligible stays...


   Sepsis ID: 100%|██████████| 73068/73068 [03:28<00:00, 350.89it/s]



   ✅ Sepsis-3 cases: 21,745
   Mean onset hour: 29.2


In [8]:
# ============================================================================
# CELL 7: CREATE MATCHED CONTROLS (1:5 ratio)
# ============================================================================

print(f"\n[6/8] Creating matched controls (1:{CONFIG['control_ratio']})...")

sepsis_full = sepsis_df.merge(
    stays_df[['stay_id', 'subject_id', 'intime', 'age', 'gender', 'los_hours']],
    on='stay_id'
)

sepsis_stay_ids = set(sepsis_df['stay_id'])
control_pool = stays_df[
    (~stays_df['stay_id'].isin(sepsis_stay_ids)) &
    (stays_df['los_hours'] >= CONFIG['min_icu_hours'])
].copy()

print(f"   Sepsis cases: {len(sepsis_full):,}")
print(f"   Control pool: {len(control_pool):,}")

used_controls = set()
all_controls = []

for _, sep_row in tqdm(sepsis_full.iterrows(), total=len(sepsis_full), desc="   Matching"):
    candidates = control_pool[
        (control_pool['age'] >= sep_row['age'] - 10) &
        (control_pool['age'] <= sep_row['age'] + 10) &
        (control_pool['los_hours'] >= sep_row['onset_hour'] + 12) &
        (~control_pool['stay_id'].isin(used_controls))
    ]

    n_match = min(CONFIG['control_ratio'], len(candidates))
    if n_match > 0:
        sampled = candidates.sample(n_match, random_state=42)
        for _, ctrl in sampled.iterrows():
            all_controls.append({
                'stay_id': ctrl['stay_id'],
                'subject_id': ctrl['subject_id'],
                'intime': ctrl['intime'],
                'age': ctrl['age'],
                'gender': ctrl['gender'],
                'onset_hour': sep_row['onset_hour'],
                'label': 0
            })
            used_controls.add(ctrl['stay_id'])

control_df = pd.DataFrame(all_controls)

sepsis_cohort = sepsis_full[['stay_id', 'subject_id', 'intime', 'age', 'gender', 'onset_hour']].copy()
sepsis_cohort['label'] = 1

cohort_df = pd.concat([sepsis_cohort, control_df], ignore_index=True)

print(f"\n   ✅ Final cohort: {len(cohort_df):,}")
print(f"   Sepsis: {(cohort_df['label']==1).sum():,} ({(cohort_df['label']==1).mean()*100:.1f}%)")
print(f"   Control: {(cohort_df['label']==0).sum():,} ({(cohort_df['label']==0).mean()*100:.1f}%)")


[6/8] Creating matched controls (1:5)...
   Sepsis cases: 21,745
   Control pool: 69,031


   Matching: 100%|██████████| 21745/21745 [04:14<00:00, 85.50it/s]



   ✅ Final cohort: 85,137
   Sepsis: 21,745 (25.5%)
   Control: 63,392 (74.5%)


In [9]:
# ============================================================================
# CELL 8: EXTRACT FEATURES
# ============================================================================

print(f"\n[7/8] Extracting features...")

FEATURE_VITALS = {
    220045: 'heart_rate', 220179: 'sbp', 220050: 'sbp',
    220180: 'dbp', 220051: 'dbp', 220052: 'map',
    220210: 'resp_rate', 220277: 'spo2',
    223761: 'temperature', 223762: 'temperature',
    220739: 'gcs_eye', 223900: 'gcs_verbal', 223901: 'gcs_motor',
}

FEATURE_LABS = {
    51301: 'wbc', 50912: 'creatinine', 51265: 'platelets',
    50885: 'bilirubin', 50813: 'lactate', 50931: 'glucose',
    51006: 'bun', 50983: 'sodium', 50971: 'potassium', 51222: 'hemoglobin',
}

ALL_FEATURES = {**FEATURE_VITALS, **FEATURE_LABS}
FEATURE_NAMES = list(set(ALL_FEATURES.values()))

cohort_stay_ids = cohort_df['stay_id'].tolist()

print("   Extracting vitals...")
vitals_dfs = []
vital_items = ','.join(map(str, FEATURE_VITALS.keys()))

for i in tqdm(range(0, len(cohort_stay_ids), CONFIG['chunk_size']), desc="   Vitals"):
    chunk = cohort_stay_ids[i:i+CONFIG['chunk_size']]
    stays_str = ','.join(map(str, chunk))
    query = f"""
    SELECT stay_id, charttime, itemid, valuenum
    FROM `physionet-data.mimiciv_3_1_icu.chartevents`
    WHERE stay_id IN ({stays_str}) AND itemid IN ({vital_items}) AND valuenum IS NOT NULL
    """
    df = client.query(query).to_dataframe()
    if len(df) > 0:
        vitals_dfs.append(df)

vitals_raw = pd.concat(vitals_dfs, ignore_index=True) if vitals_dfs else pd.DataFrame()
print(f"   Vitals: {len(vitals_raw):,} rows")

print("   Extracting labs...")
labs_dfs = []
lab_items = ','.join(map(str, FEATURE_LABS.keys()))

for i in tqdm(range(0, len(cohort_stay_ids), CONFIG['chunk_size']), desc="   Labs"):
    chunk = cohort_stay_ids[i:i+CONFIG['chunk_size']]
    stays_str = ','.join(map(str, chunk))
    query = f"""
    SELECT icu.stay_id, le.charttime, le.itemid, le.valuenum
    FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
    JOIN `physionet-data.mimiciv_3_1_icu.icustays` icu
        ON le.subject_id = icu.subject_id
        AND le.charttime BETWEEN icu.intime AND icu.outtime
    WHERE icu.stay_id IN ({stays_str}) AND le.itemid IN ({lab_items}) AND le.valuenum IS NOT NULL
    """
    df = client.query(query).to_dataframe()
    if len(df) > 0:
        labs_dfs.append(df)

labs_raw = pd.concat(labs_dfs, ignore_index=True) if labs_dfs else pd.DataFrame()
print(f"   Labs: {len(labs_raw):,} rows")

df_raw = pd.concat([vitals_raw, labs_raw], ignore_index=True)
df_raw['feature'] = df_raw['itemid'].map(ALL_FEATURES)
print(f"\n   ✅ Total measurements: {len(df_raw):,}")


[7/8] Extracting features...
   Extracting vitals...


   Vitals: 100%|██████████| 18/18 [01:27<00:00,  4.87s/it]


   Vitals: 54,300,134 rows
   Extracting labs...


   Labs: 100%|██████████| 18/18 [01:01<00:00,  3.42s/it]


   Labs: 4,907,150 rows

   ✅ Total measurements: 59,207,284


In [11]:
# ============================================================================
# CELL 9: CREATE FEATURE TENSORS (GPU-ACCELERATED)
# ============================================================================

print(f"\n[8/8] Creating feature tensors (GPU-accelerated)...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"   Using device: {device}")

N_TIMESTEPS = CONFIG['observation_window_hours']
N_FEATURES = len(FEATURE_NAMES)

# Pre-compute feature index mapping
FEATURE_TO_IDX = {fname: idx for idx, fname in enumerate(FEATURE_NAMES)}

def process_stay_tensor_gpu(stay_id, stay_data, cohort_row):
    intime = cohort_row['intime']
    onset_hour = cohort_row['onset_hour']

    end_hour = onset_hour - CONFIG['prediction_gap_hours']
    start_hour = max(0, end_hour - N_TIMESTEPS)

    if end_hour <= start_hour:
        return None

    stay_data = stay_data.copy()
    stay_data['hours'] = (stay_data['charttime'] - intime).dt.total_seconds() / 3600
    stay_data = stay_data[(stay_data['hours'] >= start_hour) & (stay_data['hours'] < end_hour)]

    if len(stay_data) < 5:
        return None

    stay_data['time_bin'] = ((stay_data['hours'] - start_hour)).astype(int).clip(0, N_TIMESTEPS-1)

    # Vectorized aggregation using pandas pivot_table (much faster than loops)
    pivot = stay_data.pivot_table(
        index='time_bin',
        columns='feature',
        values='valuenum',
        aggfunc='mean'
    )

    # Reindex to ensure all time bins exist
    pivot = pivot.reindex(range(N_TIMESTEPS))

    # Reindex columns to match FEATURE_NAMES order
    pivot = pivot.reindex(columns=FEATURE_NAMES)

    # Forward fill, backward fill, then fill remaining with 0
    pivot = pivot.ffill().bfill().fillna(0)

    # Convert to GPU tensor
    features_gpu = torch.tensor(pivot.values, device=device, dtype=torch.float32)

    # Compute derived features on GPU
    derived_list = []

    # Shock index (HR / SBP)
    if 'heart_rate' in FEATURE_TO_IDX and 'sbp' in FEATURE_TO_IDX:
        hr = features_gpu[:, FEATURE_TO_IDX['heart_rate']]
        sbp = features_gpu[:, FEATURE_TO_IDX['sbp']]
        shock_idx = torch.where(sbp != 0, hr / sbp, torch.zeros_like(hr))
        derived_list.append(shock_idx.unsqueeze(1))

    # BUN/Creatinine ratio
    if 'bun' in FEATURE_TO_IDX and 'creatinine' in FEATURE_TO_IDX:
        bun = features_gpu[:, FEATURE_TO_IDX['bun']]
        cr = features_gpu[:, FEATURE_TO_IDX['creatinine']]
        bun_cr = torch.where(cr != 0, bun / cr, torch.zeros_like(bun))
        derived_list.append(bun_cr.unsqueeze(1))

    # Concatenate derived features on GPU
    if derived_list:
        derived_tensor = torch.cat(derived_list, dim=1)
        features_gpu = torch.cat([features_gpu, derived_tensor], dim=1)

    return features_gpu.cpu().numpy()

# Parallel batch processing
cohort_lookup = cohort_df.set_index('stay_id').to_dict('index')
df_raw_grouped = df_raw.groupby('stay_id')

def process_batch(stay_ids_batch):
    batch_X, batch_y, batch_stay_ids, batch_subject_ids = [], [], [], []

    for stay_id in stay_ids_batch:
        if stay_id not in cohort_lookup:
            continue
        try:
            stay_data = df_raw_grouped.get_group(stay_id)
        except:
            continue

        cohort_row = cohort_lookup[stay_id]
        features = process_stay_tensor_gpu(stay_id, stay_data, cohort_row)

        if features is not None:
            batch_X.append(features)
            batch_y.append(cohort_row['label'])
            batch_stay_ids.append(stay_id)
            batch_subject_ids.append(cohort_row['subject_id'])

    return batch_X, batch_y, batch_stay_ids, batch_subject_ids

# Process in parallel batches
unique_stays = cohort_df['stay_id'].unique()
batch_size = 1000

X_list, y_list = [], []
stay_ids_processed, subject_ids_processed = [], []

for i in tqdm(range(0, len(unique_stays), batch_size), desc="   GPU Tensors"):
    batch = unique_stays[i:i+batch_size]
    bX, by, bstay, bsubj = process_batch(batch)
    X_list.extend(bX)
    y_list.extend(by)
    stay_ids_processed.extend(bstay)
    subject_ids_processed.extend(bsubj)

X_final = np.array(X_list, dtype=np.float32)
y_final = np.array(y_list, dtype=np.float32)
stay_ids_processed = np.array(stay_ids_processed)
subject_ids_processed = np.array(subject_ids_processed)

X_final = np.nan_to_num(X_final, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\n{'='*70}")
print("EXTRACTION COMPLETE!")
print(f"{'='*70}")
print(f"   X_final: {X_final.shape}")
print(f"   y_final: {y_final.shape}")
print(f"   Sepsis: {int(y_final.sum()):,} ({y_final.mean()*100:.1f}%)")
print(f"   Controls: {int(len(y_final) - y_final.sum()):,}")
print(f"{'='*70}")


[8/8] Creating feature tensors (GPU-accelerated)...
   Using device: cuda


   GPU Tensors: 100%|██████████| 86/86 [10:36<00:00,  7.40s/it]



EXTRACTION COMPLETE!
   X_final: (84010, 24, 22)
   y_final: (84010,)
   Sepsis: 20,874 (24.8%)
   Controls: 63,136


In [12]:
# ============================================================================
# CELL 10: SAVE DATA
# ============================================================================

import pickle

print("Saving data...")

np.save('X_final.npy', X_final)
np.save('y_final.npy', y_final)
np.save('stay_ids_processed.npy', stay_ids_processed)
np.save('subject_ids_processed.npy', subject_ids_processed)
cohort_df.to_csv('cohort_info.csv', index=False)

with open('feature_names.pkl', 'wb') as f:
    pickle.dump(FEATURE_NAMES, f)

print("\n✅ Saved:")
print("   - X_final.npy")
print("   - y_final.npy")
print("   - stay_ids_processed.npy")
print("   - subject_ids_processed.npy")
print("   - cohort_info.csv")
print("   - feature_names.pkl")

try:
    from google.colab import files
    files.download('X_final.npy')
    files.download('y_final.npy')
    files.download('cohort_info.csv')
    print("\n📥 Files downloaded!")
except:
    print("\n   Files saved locally.")

print(f"\n🎉 Ready for model training!")
print(f"   Total samples: {len(X_final):,}")

Saving data...

✅ Saved:
   - X_final.npy
   - y_final.npy
   - stay_ids_processed.npy
   - subject_ids_processed.npy
   - cohort_info.csv
   - feature_names.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 Files downloaded!

🎉 Ready for model training!
   Total samples: 84,010


In [14]:
# ============================================================================
# CELL 11: MAXIMUM GPU-OPTIMIZED MODEL TRAINING
# ============================================================================

!pip install -q lightgbm xgboost shap scikit-learn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import GradScaler, autocast  # Mixed precision
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb
import xgboost as xgb
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
import gc
warnings.filterwarnings('ignore')

# ============================================================================
# GPU OPTIMIZATION SETTINGS
# ============================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    # Enable cuDNN autotuner
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True

    # Enable TF32 for Ampere GPUs (30xx, 40xx, A100, etc.)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    # Clear cache
    torch.cuda.empty_cache()
    gc.collect()

    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   cuDNN benchmark: ENABLED")
    print(f"   TF32: ENABLED")
else:
    print("⚠️ No GPU detected, using CPU")

# ============================================================================
# DATA PREPARATION (GPU-ACCELERATED)
# ============================================================================

print("\n" + "="*70)
print("DATA PREPARATION")
print("="*70)

# Train/test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_final, y_final, groups=subject_ids_processed))

X_train, X_test = X_final[train_idx], X_final[test_idx]
y_train, y_test = y_final[train_idx], y_final[test_idx]

print(f"Train: {len(X_train):,} ({y_train.mean()*100:.1f}% sepsis)")
print(f"Test:  {len(X_test):,} ({y_test.mean()*100:.1f}% sepsis)")

# GPU-accelerated scaling
n_samples, n_timesteps, n_features = X_train.shape

X_train_tensor = torch.tensor(X_train, device=device, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, device=device, dtype=torch.float32)

# Compute RobustScaler stats on GPU
X_flat = X_train_tensor.reshape(-1, n_features)
median = torch.median(X_flat, dim=0).values
q75 = torch.quantile(X_flat, 0.75, dim=0)
q25 = torch.quantile(X_flat, 0.25, dim=0)
iqr = q75 - q25
iqr = torch.where(iqr == 0, torch.ones_like(iqr), iqr)  # Avoid div by zero

# Scale on GPU
X_train_scaled = ((X_train_tensor.reshape(-1, n_features) - median) / iqr).reshape(n_samples, n_timesteps, n_features)
X_test_scaled = ((X_test_tensor.reshape(-1, n_features) - median) / iqr).reshape(len(X_test), n_timesteps, n_features)

y_train_tensor = torch.tensor(y_train, device=device, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, device=device, dtype=torch.float32)

print(f"Features: {n_timesteps} timesteps × {n_features} features")
print(f"Data loaded to GPU ✅")

# ============================================================================
# MODEL 1: LSTM WITH ATTENTION (MAXIMUM GPU OPTIMIZATION)
# ============================================================================

print("\n" + "="*70)
print("MODEL 1: LSTM WITH ATTENTION (GPU-OPTIMIZED)")
print("="*70)

class AttentionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=256, num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout, bidirectional=True
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, 1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = F.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.classifier(context)

# Optimized DataLoaders
BATCH_SIZE = 256  # Larger batch for GPU efficiency

train_dataset = TensorDataset(X_train_scaled, y_train_tensor)
test_dataset = TensorDataset(X_test_scaled, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=False,  # Already on GPU
    drop_last=True     # Better for batch norm
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE * 2,  # Larger batch for inference
    shuffle=False
)

# Initialize model with GPU optimizations
lstm_model = AttentionLSTM(n_features).to(device)

# Use torch.compile for PyTorch 2.0+ (major speedup)
if hasattr(torch, 'compile'):
    lstm_model = torch.compile(lstm_model, mode='reduce-overhead')
    print("   torch.compile: ENABLED")

optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=0.002, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=0.01, epochs=100, steps_per_epoch=len(train_loader)
)

# Mixed precision scaler
scaler = GradScaler()
criterion = nn.BCEWithLogitsLoss()

# Training loop with mixed precision
best_auc = 0
patience_counter = 0
max_patience = 20

print(f"   Batch size: {BATCH_SIZE}")
print(f"   Mixed precision: ENABLED")
print(f"   Training...")

for epoch in range(100):
    lstm_model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()

        # Mixed precision forward pass
        with autocast():
            outputs = lstm_model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)

        # Mixed precision backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()

    # Fast GPU validation
    lstm_model.eval()
    val_preds = []
    with torch.no_grad(), autocast():
        for X_batch, _ in test_loader:
            outputs = torch.sigmoid(lstm_model(X_batch)).squeeze()
            val_preds.append(outputs)

    val_preds = torch.cat(val_preds).cpu().numpy()
    val_auc = roc_auc_score(y_test, val_preds)

    if val_auc > best_auc:
        best_auc = val_auc
        best_lstm_preds = val_preds.copy()
        patience_counter = 0
        torch.save(lstm_model.state_dict(), 'best_lstm.pt')
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0:
        print(f"   Epoch {epoch+1}: Loss={train_loss/len(train_loader):.4f}, AUC={val_auc:.4f}, LR={scheduler.get_last_lr()[0]:.6f}")

    if patience_counter >= max_patience:
        print(f"   Early stopping at epoch {epoch+1}")
        break

lstm_model.load_state_dict(torch.load('best_lstm.pt'))
lstm_auc = best_auc
lstm_preds = np.array(best_lstm_preds)
print(f"\n   ✅ LSTM AUC: {lstm_auc:.4f}")

# ============================================================================
# GPU-ACCELERATED TEMPORAL FEATURE ENGINEERING
# ============================================================================

print("\n" + "="*70)
print("GPU FEATURE ENGINEERING FOR GBM")
print("="*70)

def create_temporal_features_gpu(X_tensor):
    """Create temporal features entirely on GPU."""
    n_samples, n_timesteps, n_features = X_tensor.shape

    features_list = []

    # Statistical features (all on GPU)
    features_list.append(X_tensor.mean(dim=1))                          # mean
    features_list.append(X_tensor.std(dim=1))                           # std
    features_list.append(X_tensor.min(dim=1).values)                    # min
    features_list.append(X_tensor.max(dim=1).values)                    # max
    features_list.append(X_tensor[:, -1, :])                            # last
    features_list.append(X_tensor[:, 0, :])                             # first
    features_list.append(X_tensor[:, -1, :] - X_tensor[:, 0, :])        # trend
    features_list.append(X_tensor.median(dim=1).values)                 # median
    features_list.append(torch.quantile(X_tensor, 0.25, dim=1))         # q25
    features_list.append(torch.quantile(X_tensor, 0.75, dim=1))         # q75
    features_list.append(X_tensor.max(dim=1).values - X_tensor.min(dim=1).values)  # range

    # Temporal features
    features_list.append(X_tensor[:, -6:, :].mean(dim=1))               # last 6h mean
    features_list.append(X_tensor[:, :6, :].mean(dim=1))                # first 6h mean
    features_list.append(X_tensor[:, -6:, :].mean(dim=1) - X_tensor[:, :6, :].mean(dim=1))  # 6h change

    # Variability
    diff = X_tensor[:, 1:, :] - X_tensor[:, :-1, :]
    features_list.append(diff.abs().mean(dim=1))                        # mean absolute change
    features_list.append(diff.std(dim=1))                               # volatility

    # Concatenate all features
    features = torch.cat(features_list, dim=1)

    return features

print("   Creating features on GPU...")
X_train_gbm_gpu = create_temporal_features_gpu(X_train_scaled)
X_test_gbm_gpu = create_temporal_features_gpu(X_test_scaled)

# Move to CPU for LightGBM/XGBoost
X_train_gbm = X_train_gbm_gpu.cpu().numpy()
X_test_gbm = X_test_gbm_gpu.cpu().numpy()

# Clean NaN/Inf
X_train_gbm = np.nan_to_num(X_train_gbm, nan=0.0, posinf=0.0, neginf=0.0)
X_test_gbm = np.nan_to_num(X_test_gbm, nan=0.0, posinf=0.0, neginf=0.0)

print(f"   GBM features: {X_train_gbm.shape[1]} (created on GPU)")

# ============================================================================
# MODEL 2: LightGBM (GPU)
# ============================================================================

print("\n" + "="*70)
print("MODEL 2: LightGBM (GPU)")
print("="*70)

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'max_depth': 8,
    'learning_rate': 0.02,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'verbose': -1,
    'n_estimators': 3000,
    'early_stopping_rounds': 150,
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'gpu_use_dp': False,  # Use single precision for speed
}

lgb_train = lgb.Dataset(X_train_gbm, y_train)
lgb_test = lgb.Dataset(X_test_gbm, y_test, reference=lgb_train)

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    valid_sets=[lgb_test],
    callbacks=[lgb.log_evaluation(200)]
)

lgb_preds = lgb_model.predict(X_test_gbm)
lgb_auc = roc_auc_score(y_test, lgb_preds)
print(f"\n   ✅ LightGBM AUC: {lgb_auc:.4f}")

# ============================================================================
# MODEL 3: XGBoost (GPU)
# ============================================================================

print("\n" + "="*70)
print("MODEL 3: XGBoost (GPU)")
print("="*70)

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 8,
    'learning_rate': 0.02,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'tree_method': 'hist',
    'device': 'cuda',  # NEW: replaces gpu_id and tree_method='gpu_hist'
    'n_estimators': 3000,
    'early_stopping_rounds': 150,
    'verbosity': 0
}

xgb_model = xgb.XGBClassifier(**xgb_params)
xgb_model.fit(
    X_train_gbm, y_train,
    eval_set=[(X_test_gbm, y_test)],
    verbose=200
)

xgb_preds = xgb_model.predict_proba(X_test_gbm)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_preds)
print(f"\n   ✅ XGBoost AUC: {xgb_auc:.4f}")

# ============================================================================
# ENSEMBLE (GPU-ACCELERATED)
# ============================================================================

print("\n" + "="*70)
print("ENSEMBLE MODEL")
print("="*70)

# Weighted ensemble on GPU
preds_tensor = torch.tensor(
    np.stack([lstm_preds, lgb_preds, xgb_preds]),
    device=device, dtype=torch.float32
)

# Learn optimal weights using GPU
weights = torch.tensor([lstm_auc, lgb_auc, xgb_auc], device=device)
weights = weights / weights.sum()

ensemble_preds = (preds_tensor * weights.unsqueeze(1)).sum(dim=0).cpu().numpy()
ensemble_auc = roc_auc_score(y_test, ensemble_preds)

print(f"   Weights: LSTM={weights[0]:.3f}, LGB={weights[1]:.3f}, XGB={weights[2]:.3f}")
print(f"   ✅ Ensemble AUC: {ensemble_auc:.4f}")

# ============================================================================
# RESULTS SUMMARY
# ============================================================================

models = {
    'LSTM': lstm_auc,
    'LightGBM': lgb_auc,
    'XGBoost': xgb_auc,
    'Ensemble': ensemble_auc
}

best_model_name = max(models, key=models.get)
best_preds = {
    'LSTM': lstm_preds,
    'LightGBM': lgb_preds,
    'XGBoost': xgb_preds,
    'Ensemble': ensemble_preds
}[best_model_name]

print("\n" + "="*70)
print("🏆 RESULTS SUMMARY")
print("="*70)
for name, auc in sorted(models.items(), key=lambda x: x[1], reverse=True):
    flag = "👑" if name == best_model_name else "  "
    print(f"   {flag} {name}: AUC = {auc:.4f}")

print(f"\n   Best Model: {best_model_name}")

# GPU memory cleanup
torch.cuda.empty_cache()
gc.collect()
print("\n   GPU memory cleared ✅")

🚀 GPU: NVIDIA L4
   Memory: 23.8 GB
   cuDNN benchmark: ENABLED
   TF32: ENABLED

DATA PREPARATION
Train: 67,274 (24.8% sepsis)
Test:  16,736 (25.0% sepsis)
Features: 24 timesteps × 22 features
Data loaded to GPU ✅

MODEL 1: LSTM WITH ATTENTION (GPU-OPTIMIZED)
   torch.compile: ENABLED
   Batch size: 256
   Mixed precision: ENABLED
   Training...
   Epoch 10: Loss=1.0906, AUC=0.8401, LR=0.002801
   Epoch 20: Loss=0.9073, AUC=0.8404, LR=0.007601
   Early stopping at epoch 23

   ✅ LSTM AUC: 0.8954

GPU FEATURE ENGINEERING FOR GBM
   Creating features on GPU...
   GBM features: 352 (created on GPU)

MODEL 2: LightGBM (GPU)
[200]	valid_0's auc: 0.915493
[400]	valid_0's auc: 0.918142
[600]	valid_0's auc: 0.918687
[800]	valid_0's auc: 0.919066
[1000]	valid_0's auc: 0.919295
[1200]	valid_0's auc: 0.919516

   ✅ LightGBM AUC: 0.9196

MODEL 3: XGBoost (GPU)
[0]	validation_0-auc:0.89551
[200]	validation_0-auc:0.91674
[400]	validation_0-auc:0.91914
[600]	validation_0-auc:0.91973
[800]	validation